## Prerequisites
- pip install langchain langchain_community langchain_chroma
- pip install -qU langchain-openai
- export LANGCHAIN_API_KEY="..."

In [1]:
import os

os.environ['USER_AGENT'] = 'sports-buddy-basic'

In [2]:
# Initialize an OpenAI model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []
loader = WebBaseLoader(
  web_paths=("https://en.wikipedia.org/wiki/2024_Summer_Olympics",),
)
docs = loader.load()

In [ ]:
# TODO: Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# TODO: Store documents in Chroma vector database
database = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# Retrieve documents.
retriever = database.as_retriever()
prompt = hub.pull("rlm/rag-prompt")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

    rag_chain.invoke("Which programmes were dropped from the 2024 Olympics?")

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
  {"context": retriever | format_docs, "question": RunnablePassthrough()}
  | prompt
  | llm
  | StrOutputParser()
)

In [ ]:
rag_chain.invoke("Which programmes were dropped from the 2024 Olympics?")